# LoRA: teach a decoder one exact label

Colab GPU. Data: `dair-ai/emotion`. Model: `Qwen/Qwen2.5-0.5B-Instruct`. Restart once after the install cell. Save the adapter to Drive. Do not upload it.


**PulseDesk** tags each post with one word: sadness, joy, love, anger, fear, or surprise. A sentence is rejected. These are dataset labels, not a diagnosis.

```text
post → prompt → frozen Qwen + small LoRA adapter → one word, or INVALID
```


## 1. Setup

Pin the libraries used in all three notebooks. Restart Colab once after the install. The cell removes an old optional package, `torchao`, because it can stop PEFT from attaching LoRA. We do not use it.


In [ ]:
# Install the packages used by the two-lesson sequence.

# torchao is optional here; Colab may ship a version too old for PEFT.
%pip -q uninstall -y torchao
%pip -q install -U "transformers==5.17.0" "datasets==5.0.1" "trl==0.28.0" "peft==0.21.0" accelerate scikit-learn


In [ ]:
# Check device and record library versions.

import json, random, importlib.metadata as metadata
from pathlib import Path
import numpy as np
import torch
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), "Select a GPU in Colab: Runtime → Change runtime type → GPU, then rerun."
print('GPU:', torch.cuda.get_device_name(0))
for package in ('torch', 'transformers', 'datasets', 'trl', 'peft'):
    print(package, metadata.version(package))
import trl
assert trl.__version__ == '0.28.0', (
    f'Loaded TRL {trl.__version__}; restart the Colab runtime after the install cell, then rerun from the top.'
)


## 2. PEFT, then LoRA

**PEFT** means train a small add-on and leave the pretrained weights frozen. **LoRA** is one PEFT method: two small matrices beside a frozen weight. **QLoRA** is that same add-on with the frozen weight stored in 4 bits. This notebook trains plain LoRA.

The adapter is not a whole model. `LoraConfig` only says where it goes. The trainer does the update.

| Ticket | First move |
|---|---|
| Three posts, need an answer before lunch | Use a prompt. Do not train. |
| Every row must be one of six words, and we have many labels | Fine-tune, then compare with the base model. This is today's job. |
| The answer changes when a policy PDF changes | Look up the current PDF. Training does not refresh facts. |

PEFT is the family. LoRA is one member, so the two names are not interchangeable.


### Where the adapter sits

Inside each decoder block, attention has `q_proj`, `k_proj`, `v_proj`, `o_proj`. The MLP has `gate_proj`, `up_proj`, `down_proj`. Those are weight matrices, so LoRA can sit beside them. A GELU or a residual add cannot. The same idea works on other models, but the names change, so we check Qwen in code.

```text
q_proj → what this token looks for
k_proj → what earlier tokens can match
v_proj → the information that gets mixed in
```

| Targets | Choice |
|---|---|
| `q_proj` and `v_proj` | Small and easy to see. **This run.** |
| all four attention projections | More capacity. A later experiment. |
| `all-linear` | Also adapts the MLP. More memory. |

Two target names do not mean two matrices in the whole model. Every block gets its own pair. LoRA on `v_proj` can change the information attention mixes. LoRA on `q_proj` can change which earlier tokens get weight.


### The update

One frozen projection `W`, hidden state `h` (a vector, not the word):

```text
output = W h + (alpha / r) B (A h)
```

`W` stays frozen. `A` and `B` learn. Rank `r` is the width of the small path. We use `r=8` and `alpha=16`, so the correction is multiplied by 2.

A 1,024×1,024 matrix has about 1 million values. Rank 8 adds 16,384 values, about **1.6% of that one matrix**, not of the whole model. The trainer prints the real fraction later. The base weights are still on the GPU. Freezing only stops their update.


### Follow four numbers

This toy uses rank 1, so `A` is one row and `B` is one column. `A` compresses 4 numbers into 1. `B` expands that 1 number into a correction. We add the correction. We do not replace `W`. The identity matrix is only so the original output is easy to see. Real `q_proj` is not the identity, and the real run uses rank 8.


In [ ]:
# Follow one token's hidden state through a toy q_proj and its LoRA path.

# These A and B values are for hand math. PEFT usually starts the adapter at zero, then training moves it.

A_q = torch.tensor([[1.0, 0.0, -1.0, 2.0]])       # down-projection: 4 hidden values -> rank 1
B_q = torch.tensor([[0.1], [0.0], [-0.2], [0.3]])  # up-projection: that 1 value -> a 4-number correction
weight_update_q = B_q @ A_q                         # (4, 1) @ (1, 4) = (4, 4). Rank 1: one shared direction.
print("A_q shape (rank, in):", tuple(A_q.shape), "| B_q shape (out, rank):", tuple(B_q.shape))
print("B_q @ A_q shape:", tuple(weight_update_q.shape))
print("B_q @ A_q =\n", weight_update_q)
print("Each row is A_q scaled by one entry of B_q. That is the whole rank-1 update.")

h = torch.tensor([1.0, 2.0, 0.0, -1.0])             # one token's hidden state, not the word itself
z = A_q @ h                                         # A compresses h into one number
query_correction = B_q @ z                          # expand -1 through B_q
W_q = torch.eye(4)                                  # teaching choice so W_q h = h. Real q_proj is learned, not I.
frozen_query = W_q @ h
scale = 1.0                                         # this toy uses alpha/r = 1. The training run uses 16/8 = 2.
adapted_query = frozen_query + scale * query_correction
print("hidden state h:", h.tolist())
print("intermediate z = A_q h:", z.item())
print("LoRA correction = B_q z:", query_correction.tolist())
print("frozen query = W_q h:", frozen_query.tolist())
print("adapted query = W_q h + (alpha/r) B_q(A_q h):", adapted_query.tolist())

width, rank = 1024, 8
full_values = width * width                         # what a full update of this one matrix would touch
adapter_values = rank * width + width * rank        # A is (r, in), B is (out, r)
print("For one 1024x1024 matrix: full values =", full_values)
print("Rank-8 adapter values =", adapter_values)
print("Adapter fraction for THAT matrix =", f"{100 * adapter_values / full_values:.2f}%")
print("This percent is not the whole model. The trainer prints the real fraction after wrapping Qwen.")
print("A separate A,B pair would sit on v_proj. A_q is not A_v.")


About 1.6% of one wide matrix is not the trainable fraction of the whole model. Only the targeted layers get adapters. The next print is the real fraction.


### From the drawing to the code

Load Qwen, describe `A` and `B` with `LoraConfig`, let the trainer update only those, then save the small folder.

The names that learn are `lora_A` and `lora_B`. The original weights stay frozen.


In [ ]:
# Inspect a LoRA configuration before loading the base model.

theory_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=['q_proj','v_proj'], bias='none', task_type='CAUSAL_LM',
)
print('PEFT method:', theory_config.peft_type)
print('task:', theory_config.task_type)
print('rank:', theory_config.r, 'alpha:', theory_config.lora_alpha)
print('targets:', theory_config.target_modules)
print('No model has been modified yet; this is only a configuration object.')


## 3. What gets graded

The decoder predicts the next token from earlier tokens only. The prompt is visible. The label is what we grade.

```text
prompt: I got the job!     visible, loss skipped with -100
answer: joy + end token    graded
```

Three different tools: the causal mask hides the future, padding hides empty slots, and `-100` skips the loss. `-100` does not hide the prompt.

While predicting `joy`, the model cannot see the end token, because that token is still in the future. It can still read the post.


Row C can see A, B, and C. D is still in the future, so the causal mask hides it.


In [ ]:
# Print a four-position causal attention mask.

positions = ["A", "B", "C", "D"]  # stand-ins for four tokens, left to right in time
print("query  sees keys ->")
for i, token in enumerate(positions):
    # j <= i means "this key is the current token or an earlier one."
    visible = ["✓" if j <= i else "·" for j in range(len(positions))]
    print(f"{token}      {' '.join(visible)}")
print("Token C can use A, B, C. D is the future, so the causal mask hides it.")
print("This mask is not the loss mask. A prompt token can be visible here and still be labelled -100.")


### Loss is a probability

One graded token contributes `-ln(P(correct))`. Probability 0.60 gives a loss near 0.51. Probability 0.10 gives a loss near 2.30. Lower means the right token was more likely. Prompt tokens are not in the average.

A falling training loss means the seen answer tokens became more likely. It does not mean new posts are solved.


In [ ]:
# Turn two made-up answer-token probabilities into a supervised loss.

import math

def token_loss(probability):
    """Negative log probability of the single correct next token."""
    return -math.log(probability)

# Imagine the graded completion is two tokens: the label, then the end token.
label_loss = token_loss(0.60)   # model is fairly sure
end_loss = token_loss(0.80)
unsure_loss = token_loss(0.10)  # model almost missed the label
print(f"P(label)=0.60 -> loss {label_loss:.3f}")
print(f"P(end)=0.80   -> loss {end_loss:.3f}")
print(f"average over the two graded tokens -> {(label_loss + end_loss) / 2:.3f}")
print(f"P(label)=0.10 -> loss {unsure_loss:.3f}  (worse: the right token was unlikely)")
print("A prompt position labelled -100 is not in this average. Attention can still read it.")
print("A falling average on training posts is not yet a PulseDesk win. That check is the next lesson.")


## 4. Real posts

The emotion dataset has train, validation, and test. We fit on 1,200 train rows and watch 240 validation rows. Test waits for the evaluation notebook. The integer label becomes a word such as `joy`.

Class shares matter because a common label can make accuracy look kind while a rare emotion is ignored. Macro-F1 gives each emotion an equal vote.


In [ ]:
# Load official splits, label names, and small classroom subsets.

DATASET_ID, DATASET_CONFIG = 'dair-ai/emotion', 'split'
raw = load_dataset(DATASET_ID, DATASET_CONFIG)
LABELS = list(raw['train'].features['label'].names)
assert set(raw) >= {'train', 'validation', 'test'}
assert set(LABELS) == {'sadness', 'joy', 'love', 'anger', 'fear', 'surprise'}
TRAIN_ROWS, VAL_ROWS = 1200, 240
train_raw = raw['train'].shuffle(seed=SEED).select(range(TRAIN_ROWS))
val_raw = raw['validation'].shuffle(seed=SEED).select(range(VAL_ROWS))
print({part: len(raw[part]) for part in ('train','validation','test')})
print('Labels:', LABELS)
for row in train_raw.select(range(3)):
    print({'text': row['text'], 'label': LABELS[row['label']]})


In [ ]:
# Count examples by label in the selected train and validation subsets.

from collections import Counter
for name, part in [('train subset', train_raw), ('validation subset', val_raw)]:
    counts = Counter(part['label'])
    print(name, {label: counts[i] for i, label in enumerate(LABELS)})


Name the biggest share and the smallest share from the print.


In [ ]:
# Show each class as a percentage of the selected subsets.

for name, part in [('train subset', train_raw), ('validation subset', val_raw)]:
    counts = Counter(part['label'])
    print(name)
    for i, label in enumerate(LABELS):
        print(f'  {label:9} {counts[i]:4d} / {len(part)} = {counts[i]/len(part):.1%}')


Read one real post per label. Some wording fits two emotions. The dataset label is still what we grade. Point at one ambiguous post.


In [ ]:
# Show one real selected training post for each of the six labels.

seen = set()
for row in train_raw:
    label = LABELS[row['label']]
    if label not in seen:
        print(f'{label:9} | {row["text"]}')
        seen.add(label)
    if len(seen) == len(LABELS):
        break
assert len(seen) == len(LABELS)


## 5. Prompt and answer

The prompt stops where the assistant starts. The completion is the label plus the end token. The same prompt is used later, when the model generates.

```text
A. prompt = instruction + post            completion = joy
B. prompt = instruction + post + "joy"    completion = joy
C. prompt = instruction + post            completion = joy + end token
```

B leaks the answer into the context, so the model can copy it. C is the contract this lab trains: the label and the end token are the graded completion. A is incomplete unless the trainer adds the end token. The next cell builds C with Qwen's chat template.


In [ ]:
# Load the small instruction-tuned decoder and inspect its chat template.

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
SYSTEM_INSTRUCTION = (
    'Classify the emotion expressed in the text. '
    'Choose exactly one label: ' + ', '.join(LABELS) + '. '
    'Return only the label, with no explanation.'
)
def prompt_for(text):
    messages = [
        {'role':'system', 'content':SYSTEM_INSTRUCTION},
        {'role':'user', 'content':'Text: ' + text},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print('eos token:', repr(tokenizer.eos_token), 'pad token:', repr(tokenizer.pad_token))
print(prompt_for(train_raw[0]['text'])[-500:])


In [ ]:
# Convert real rows to prompt/completion records.

MAX_LENGTH = 256
def to_sft(part):
    records = {'prompt': [], 'completion': []}
    dropped = 0
    for row in part:
        prompt = prompt_for(row['text'])
        completion = LABELS[row['label']] + tokenizer.eos_token
        length = len(tokenizer(prompt + completion, add_special_tokens=False)['input_ids'])
        if length > MAX_LENGTH:
            dropped += 1
            continue
        records['prompt'].append(prompt)
        records['completion'].append(completion)
    return Dataset.from_dict(records), dropped

train_sft, train_dropped = to_sft(train_raw)
val_sft, val_dropped = to_sft(val_raw)
assert len(train_sft) > 0 and len(val_sft) > 0
print('train/validation kept:', len(train_sft), len(val_sft))
print('long rows excluded:', train_dropped, val_dropped)
print('prompt ending:', repr(train_sft[0]['prompt'][-80:]))
print('completion:', repr(train_sft[0]['completion']))


Prompt tokens should say ignored. The label and the end token should say graded. Ignored means skip the loss, not "the model cannot read this."


In [ ]:
# Display token-level context versus supervised answer targets.

example = train_sft[0]
prompt_ids = tokenizer(example["prompt"], add_special_tokens=False)["input_ids"]
full_ids = tokenizer(example["prompt"] + example["completion"], add_special_tokens=False)["input_ids"]
assert full_ids[:len(prompt_ids)] == prompt_ids  # completion is appended; it does not rewrite the prompt
# -100 is the loss ignore index. It is not an attention mask and not a token the model is asked to emit as the prompt.
loss_labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
for token, target in zip(tokenizer.convert_ids_to_tokens(full_ids[-12:]), loss_labels[-12:]):
    role = "ignored (context only)" if target == -100 else "graded (answer token)"
    print(f"{token!r:24} -> {role}")
print("prompt tokens:", len(prompt_ids), "| graded tokens:", sum(x != -100 for x in loss_labels))
print("TRL builds this same idea from the prompt and completion fields. Do not shift labels a second time.")


## 6. What the frozen model already says

Same 12 validation posts, same prompt, greedy decoding. `The emotion is joy` is invalid. This is a before-picture, not the final test.


In [ ]:
# Load the base model in a GPU-friendly dtype.

USE_BF16 = bool(torch.cuda.is_bf16_supported())
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE)
model.to('cuda')
model.eval()
print('dtype:', DTYPE, 'parameters:', f'{sum(p.numel() for p in model.parameters()):,}')


Confirm `q_proj` and `v_proj` exist, then compare how many adapter values three target choices would add. We train the small one.


In [ ]:
# Map Qwen linear modules and show where LoRA could attach.

from collections import Counter
from torch import nn
print('architecture:', model.config.model_type)
print('decoder blocks:', len(model.model.layers))
linear_modules = [(name, module) for name,module in model.named_modules() if isinstance(module, nn.Linear)]
print('linear module types and counts:', Counter(name.rsplit('.',1)[-1] for name,_ in linear_modules))
first_block = [(name, tuple(module.weight.shape)) for name,module in linear_modules if '.layers.0.' in name]
for name, shape in first_block:
    print(name, shape)
assert {'q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'} <= set(
    name.rsplit('.',1)[-1] for name,_ in linear_modules
)


Rank `r` on a layer adds `r × (inputs + outputs)` adapter values. This is a parameter count, not peak GPU memory.


In [ ]:
# Estimate LoRA matrix counts for three target-module choices.

def adapter_matrix_count(target_names, rank=8):
    # A linear map of shape (out, in) gets A of shape (rank, in) and B of shape (out, rank).
    chosen = [(name, module) for name, module in linear_modules if name.rsplit(".", 1)[-1] in target_names]
    values = sum(rank * (module.in_features + module.out_features) for _, module in chosen)
    return len(chosen), values

attention_qv = {"q_proj", "v_proj"}                                  # this lesson
attention_all = {"q_proj", "k_proj", "v_proj", "o_proj"}             # broader attention
attention_mlp = attention_all | {"gate_proj", "up_proj", "down_proj"}  # attention and MLP
for title, targets in [
    ("Q and V", attention_qv),
    ("all attention", attention_all),
    ("attention + MLP", attention_mlp),
]:
    module_count, values = adapter_matrix_count(targets)
    print(f"{title:17} modules={module_count:3d}  rank-8 values={values:,}")
print("The trainer below uses Q and V. The evaluation lesson only shows all-linear inside the optional QLoRA demo.")
print("This count is adapter values, not peak GPU memory. Activations and the frozen base are still there.")


In [ ]:
# Generate a short label from the frozen base model.

def generate_one(active_model, text):
    encoded = tokenizer(prompt_for(text), return_tensors='pt').to('cuda')
    with torch.inference_mode():
        output = active_model.generate(
            **encoded, max_new_tokens=8, do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = output[0, encoded['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

preview_rows = raw['validation'].shuffle(seed=SEED).select(range(12))
base_preview = [generate_one(model, row['text']) for row in preview_rows]
for row, answer in zip(preview_rows, base_preview):
    print({'text':row['text'][:90], 'gold':LABELS[row['label']], 'base_raw':answer})


In [ ]:
# Compute preview accuracy and valid-label rate on 12 validation rows.

def strict_label(answer):
    # Trim and lowercase only. Do not search inside a sentence for a label word.
    candidate = answer.strip().lower()
    return candidate if candidate in LABELS else "INVALID"

preview_gold = [LABELS[row["label"]] for row in preview_rows]
preview_pred = [strict_label(answer) for answer in base_preview]
correct = sum(gold == pred for gold, pred in zip(preview_gold, preview_pred))
valid = sum(pred != "INVALID" for pred in preview_pred)
print("correct:", correct, "/", len(preview_gold))
print("valid:", valid, "/", len(preview_pred))
print("invalid raw answers:", [raw for raw, pred in zip(base_preview, preview_pred) if pred == "INVALID"])
print("Invalid stays in the denominator. Dropping it would make the preview look healthier than the product.")


## 7. Train

60 steps is a time limit, not a promise. Batch 2, accumulated 8 times: one update sees 16 posts, not 16 at once in memory. Grade the answer only.

Every trainable name should contain `lora_`. If an original weight is trainable, stop. That run is not this experiment.


In [ ]:
# Attach LoRA and set a bounded training schedule.

lora_config = LoraConfig(
    r=8,                 # bottleneck width: 8 intermediate values on each adapter
    lora_alpha=16,       # the correction is multiplied by alpha/r = 16/8 = 2
    lora_dropout=0.05,   # dropout on the adapter path during training
    target_modules=["q_proj", "v_proj"],  # every query and value projection, in every block
    bias="none",         # ordinary bias vectors stay frozen
    task_type="CAUSAL_LM",
)
training_args = SFTConfig(
    output_dir="/content/emotion_lora_training",
    max_length=MAX_LENGTH,            # sequences longer than this were already dropped
    completion_only_loss=True,        # grade the label; keep the prompt as context
    packing=False,                    # one post per sequence, so the mask stays easy to inspect
    per_device_train_batch_size=2,    # small, so activations fit
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,    # one optimizer step sees 2 * 8 = 16 posts
    max_steps=60,                     # a class clock, not an epoch and not a quality target
    learning_rate=2e-4,               # step size for the adapter weights
    warmup_steps=3,                   # ramp that step size over the first 3 of 60 updates
    eval_strategy="steps",
    eval_steps=20,                    # validation loss at steps 20, 40, and 60
    logging_steps=10,
    save_strategy="no",               # the Drive cell is the save that matters
    fp16=not USE_BF16,
    bf16=USE_BF16,                    # match the dtype the GPU actually supports
    gradient_checkpointing=True,      # recompute some activations to save memory
    optim="adamw_torch",
    seed=SEED,
    report_to="none",
    eos_token=tokenizer.eos_token,
    pad_token=tokenizer.pad_token,
)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_sft,
    eval_dataset=val_sft,
    peft_config=lora_config,          # the plan from the board; the trainer inserts the matrices
    processing_class=tokenizer,
)
trainer.model.print_trainable_parameters()


Read the printed fraction. If a base weight is trainable, stop.


In [ ]:
# List a few trainable parameter names and verify base weights are frozen.

trainable = [(name, p.numel()) for name,p in trainer.model.named_parameters() if p.requires_grad]
total = sum(p.numel() for p in trainer.model.parameters())
assert trainable and all('lora_' in name for name,_ in trainable)
print('trainable / total:', f'{sum(n for _,n in trainable):,}', '/', f'{total:,}')
print('trainable percentage:', f'{100*sum(n for _,n in trainable)/total:.3f}%')
print('first adapter parameters:', trainable[:8])


In [ ]:
# Train and display short training/validation loss logs.

train_result = trainer.train()
print('final training metrics:', train_result.metrics)
for log in trainer.state.log_history:
    if 'loss' in log or 'eval_loss' in log:
        print({key: log[key] for key in ('step','loss','eval_loss') if key in log})


Train loss is fit on seen batches. Validation loss is a separate split. A short curve can wobble.


In [ ]:
# Plot training and validation loss versus optimizer step.

import matplotlib.pyplot as plt
train_points = [(x['step'],x['loss']) for x in trainer.state.log_history if 'loss' in x]
val_points = [(x['step'],x['eval_loss']) for x in trainer.state.log_history if 'eval_loss' in x]
plt.figure(figsize=(7,3.5))
if train_points: plt.plot(*zip(*train_points), marker='o', label='train loss')
if val_points: plt.plot(*zip(*val_points), marker='s', label='validation loss')
plt.xlabel('optimizer step'); plt.ylabel('answer-token loss'); plt.grid(alpha=.3); plt.legend(); plt.show()


A falling training loss can still be a bad product. The adapter may have memorized practice posts, started writing sentences, or helped a common label and hurt a rare one. The 12 posts below are a glance. The test set is the later measurement.


In [ ]:
# Compare the adapted model on the SAME twelve preview rows.

trainer.model.eval()
adapted_preview = [generate_one(trainer.model, row['text']) for row in preview_rows]
for row, before, after in zip(preview_rows, base_preview, adapted_preview):
    print({'gold':LABELS[row['label']], 'base':before, 'adapter':after, 'text':row['text'][:80]})


In [ ]:
# Count base-to-adapter correctness changes on preview rows.

adapted_preview_pred = [strict_label(x) for x in adapted_preview]
improved = sum(b != g and a == g for g,b,a in zip(preview_gold,preview_pred,adapted_preview_pred))
regressed = sum(b == g and a != g for g,b,a in zip(preview_gold,preview_pred,adapted_preview_pred))
print('validation preview improvements:', improved, 'regressions:', regressed)
print('Not final: the evaluation lesson uses untouched test rows.')


## 8. Save

Colab forgets the runtime. The Drive folder is the update plus the recipe, not a copy of Qwen.

The folder cannot tag a post by itself. The same Qwen model has to load, and the adapter attaches to it.


In [ ]:
# Mount Drive and choose one persistent lesson folder.

from google.colab import drive
drive.mount('/content/drive')
ARTIFACT_DIR = Path('/content/drive/MyDrive/LogicMojo/decoder_lora_emotion')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(ARTIFACT_DIR)


In [ ]:
# Save LoRA adapter, tokenizer, and experiment settings.

trainer.model.save_pretrained(ARTIFACT_DIR, safe_serialization=True)
tokenizer.save_pretrained(ARTIFACT_DIR)
manifest = {
    'model_id': MODEL_ID, 'dataset_id': DATASET_ID, 'dataset_config': DATASET_CONFIG,
    'labels': LABELS, 'system_instruction': SYSTEM_INSTRUCTION,
    'seed': SEED, 'train_rows_requested': TRAIN_ROWS, 'val_rows_requested': VAL_ROWS,
    'train_rows_kept': len(train_sft), 'val_rows_kept': len(val_sft),
    'max_length': MAX_LENGTH, 'max_steps': training_args.max_steps,
    'lora_r': lora_config.r, 'lora_alpha': lora_config.lora_alpha,
    'target_modules': list(lora_config.target_modules),
    'package_versions': {p: metadata.version(p) for p in ('torch','transformers','datasets','trl','peft')},
}
(ARTIFACT_DIR / 'lesson_manifest.json').write_text(json.dumps(manifest, indent=2))
print(sorted(p.name for p in ARTIFACT_DIR.iterdir()))
assert (ARTIFACT_DIR / 'adapter_config.json').exists()
assert any(ARTIFACT_DIR.glob('adapter_model.*'))


The files should be megabytes. The next notebook needs `adapter_config.json` and `lesson_manifest.json`.


In [ ]:
# List saved artifact sizes after the Drive save.

for item in sorted(ARTIFACT_DIR.iterdir()):
    if item.is_file():
        print(f'{item.name:35} {item.stat().st_size/1024**2:8.3f} MiB')
